# Operaciones con Matrices y Estados Cuánticos
### Enrique Prada Vázquez

Este cuaderno contiene la resolución del Trabajo sobre álgebra lineal aplicada y manipulación de estados cuánticos utilizando las librerías **NumPy**, **cmath** y **Qiskit**.

## 0. Preparación del Entorno
En esta sección se instalan las dependencias necesarias: `numpy` para cálculo numérico, `qiskit` para computación cuántica, `pylatexenc` para la visualización de fórmulas y `sympy` para soporte simbólico.

*Después de la instalación es necesario reiniciar el kernel*

In [1]:
pip install numpy qiskit pylatexenc sympy

Note: you may need to restart the kernel to use updated packages.


## 1. Definición de Matrices
Se definen las matrices $A, B, C \in \mathbb{C}^{n \times n}$ y $D \in \mathbb{C}^{2 \times 2}$ proporcionadas en el enunciado. En Python, la unidad imaginaria se representa mediante el sufijo `j`.

In [2]:
import numpy as np
import cmath
from qiskit.quantum_info import Statevector
from IPython.display import display

# Definición de matrices
A = np.array([[3+2j, 1-1j, 4-3j], [2+1j, -5+4j, 6], [-3+2j, 7-1j, 8+3j]])
B = np.array([[2+3j, 4, 1+2j], [3-2j, 5+1j, 7-3j], [6+4j, -2, 9+5j]])
C = np.array([[1+4j, 2-2j, 3+1j, -5], [-2+3j, 6, 4+7j, 2+1j], 
              [-1+2j, -6+5j, -8j, 7+4j], [-3-4j, 3-1j, -5-3j, 5j]])
D = np.array([[1j, 2], [2-3j, 4+5j]])

display("A", A)
display("B", B)
display("C", C)
display("D", D)

'A'

array([[ 3.+2.j,  1.-1.j,  4.-3.j],
       [ 2.+1.j, -5.+4.j,  6.+0.j],
       [-3.+2.j,  7.-1.j,  8.+3.j]])

'B'

array([[ 2.+3.j,  4.+0.j,  1.+2.j],
       [ 3.-2.j,  5.+1.j,  7.-3.j],
       [ 6.+4.j, -2.+0.j,  9.+5.j]])

'C'

array([[ 1.+4.j,  2.-2.j,  3.+1.j, -5.+0.j],
       [-2.+3.j,  6.+0.j,  4.+7.j,  2.+1.j],
       [-1.+2.j, -6.+5.j, -0.-8.j,  7.+4.j],
       [-3.-4.j,  3.-1.j, -5.-3.j,  0.+5.j]])

'D'

array([[0.+1.j, 2.+0.j],
       [2.-3.j, 4.+5.j]])

## 2. Determinantes y Trazas en Forma Polar
A continuación, calculamos el **determinante** (que representa el factor de escala de volumen de la transformación) y la **traza** (suma de los elementos de la diagonal principal). Los resultados se convierten a **forma polar** $(r, \theta)$ utilizando el módulo `cmath`.

In [3]:
matrices = {'A': A, 'B': B, 'C': C, 'D': D}

for nombre, M in matrices.items():
    det = np.linalg.det(M)
    traza = np.trace(M)
    print(f"Matriz {nombre}:")
    print(f"  Det (polar): {cmath.polar(det)}")
    print(f"  Traza (polar): {cmath.polar(traza)}")

Matriz A:
  Det (polar): (248.07257002740147, -3.1174038239309567)
  Traza (polar): (10.816653826391969, 0.982793723247329)
Matriz B:
  Det (polar): (211.35751701796656, 1.056345013735869)
  Traza (polar): (18.35755975068582, 0.5123894603107377)
Matriz C:
  Det (polar): (5502.233182990338, 2.745864851140304)
  Traza (polar): (7.0710678118654755, 0.14189705460416394)
Matriz D:
  Det (polar): (13.453624047073713, 2.3036114285814033)
  Traza (polar): (7.211102550927978, 0.982793723247329)


## 3. Operación Matricial Combinada
Se realiza el cálculo de la expresión compleja:
$$2i(\det(D) + \text{tr}(C))AB - (1+i)\frac{\det(C)}{\text{tr}(D)}BA$$
Esta operación requiere el uso del operador `@` para el producto matricial, asegurando que se respete la propiedad no conmutativa de las matrices.

In [4]:
term1 = 2j * (np.linalg.det(D) + np.trace(C)) * (A @ B)
term2 = (1+1j) * (np.linalg.det(C) / np.trace(D)) * (B @ A)
resultado = term1 - term2
display("Resultado de la operación compleja:", resultado)

'Resultado de la operación compleja:'

array([[  7945.76923077+10748.84615385j,  12546.        +30202.j        ,
         52329.92307692 -3584.38461538j],
       [ 23914.        +21313.j        ,   6051.07692308-27248.61538462j,
         81557.53846154-75285.30769231j],
       [-13493.        +40279.j        ,  88230.15384615-38881.23076923j,
        109709.30769231 +8150.53846154j]])

## 4. Clasificación y Análisis de Matrices
Para clasificar las matrices, verificamos las siguientes propiedades mediante comparaciones numéricas (`np.allclose`):
* **Hermitiana:** $M = M^\dagger$
* **Unitaria:** $M^\dagger M = I$
* **Normal:** $M^\dagger M = M M^\dagger$
* **Inversa:** Se comprueba si la matriz es no singular (determinante distinto de cero).

In [5]:
def clasificar_y_mostrar(matrices):
    # Cabecera de la tabla manual
    header = f"{'Matriz':<8} | {'Hermitiana':<12} | {'Unitaria':<10} | {'Normal':<8} | {'Inversa'}"
    print(header)
    print("-" * len(header))

    for nombre, M in matrices.items():
        # Cálculos de álgebra lineal
        M_dag = M.conj().T
        identidad = np.eye(M.shape[0])
        
        # Verificaciones
        es_hermitiana = np.allclose(M, M_dag)
        es_unitaria = np.allclose(M_dag @ M, identidad)
        es_normal = np.allclose(M_dag @ M, M @ M_dag)
        
        # Inversa
        if np.linalg.det(M) == 0:
             inversa_str = "No existe"
        else:
            inversa_str = "Existe"
        
        # Formateo de cada fila
        # Usamos 'SI' y 'NO' para que sea intuitivo al leer
        print(f"{nombre:<8} | "
              f"{'SI' if es_hermitiana else 'NO':<12} | "
              f"{'SI' if es_unitaria else 'NO':<10} | "
              f"{'SI' if es_normal else 'NO':<8} | "
              f"{inversa_str}")

# Ejecutamos la función
clasificar_y_mostrar(matrices)

Matriz   | Hermitiana   | Unitaria   | Normal   | Inversa
---------------------------------------------------------
A        | NO           | NO         | NO       | Existe
B        | NO           | NO         | NO       | Existe
C        | NO           | NO         | NO       | Existe
D        | NO           | NO         | NO       | Existe


## 5. Representación de Estados en $\mathbb{C}^2$
Entramos en el bloque de computación cuántica utilizando la clase `Statevector` de Qiskit. Definimos la base computacional estándar de un qubit:
* $|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}$
* $|1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$

In [6]:
zero = Statevector([1, 0])  # |0>
one = Statevector([0, 1])   # |1>

## 6. Estados de Superposición y Estados de Bell

En esta sección, exploramos la creación de estados cuánticos que van más allá de la base computacional simple, utilizando la clase `Statevector` de Qiskit.

### 6.1. Estados de un Qubit
A partir de la base computacional $\{|0\rangle, |1\rangle\}$, generamos los estados $|+\rangle, |-\rangle, |i+\rangle, |i-\rangle$. Estos representan superposiciones equitativas en la esfera de Bloch:
* $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$
* $|-\rangle = \frac{1}{\sqrt{2}}(|0\rangle - |1\rangle)$.
* $|i+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + i|1\rangle)$
* $|i-\rangle = \frac{1}{\sqrt{2}}(|0\rangle - i|1\rangle)$.

In [7]:
# --- Estados de un qubit ---
plus = (zero + one) / np.sqrt(2)
minus = (zero - one) / np.sqrt(2)
i_plus = (zero + 1j*one) / np.sqrt(2)
i_minus = (zero - 1j*one) / np.sqrt(2)

# Visualización
print('Estado |+>')
display(plus.draw('latex'))
print('Estado |->')
display(minus.draw('latex'))
print('Estado |i+>')
display(i_plus.draw('latex'))
print('Estado |i->')
display(i_minus.draw('latex'))

Estado |+>


<IPython.core.display.Latex object>

Estado |->


<IPython.core.display.Latex object>

Estado |i+>


<IPython.core.display.Latex object>

Estado |i->


<IPython.core.display.Latex object>

### 6.2. Estados de Bell
Los estados de Bell son cuatro estados específicos de dos qubits que representan el entrelazamiento máximo. En estos estados, el sistema no puede describirse como el producto de estados individuales de cada qubit. La base de Bell se define como:

* **Estado $|\Phi^+\rangle$:** $\frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$
* **Estado $|\Phi^-\rangle$:** $\frac{1}{\sqrt{2}}(|00\rangle - |11\rangle)$
* **Estado $|\Psi^+\rangle$:** $\frac{1}{\sqrt{2}}(|01\rangle + |10\rangle)$
* **Estado $|\Psi^-\rangle$:** $\frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)$

Los resultados a continuación se muestran en formato LaTeX para verificar la correcta normalización y los coeficientes de amplitud obtenidos.

In [8]:
# --- Estados de Bell (Dos qubits) ---
# Usamos el producto tensorial para crear los estados de la base computacional de 2 qubits
z2 = zero.tensor(zero) # |00>
o2 = one.tensor(one)   # |11>
zo = zero.tensor(one)  # |01>
oz = one.tensor(zero)  # |10>

phi_plus  = (z2 + o2) / np.sqrt(2)  # |00> + |11>
phi_minus = (z2 - o2) / np.sqrt(2)  # |00> - |11>
psi_plus  = (zo + oz) / np.sqrt(2)  # |01> + |10>
psi_minus = (zo - oz) / np.sqrt(2)  # |01> - |10>

# Visualización
print('ESTADOS DE BELL')
print('Estado Phi+:')
display(phi_plus.draw('latex'))

print('Estado Phi-:')
display(phi_minus.draw('latex'))

print('Estado Psi+:')
display(psi_plus.draw('latex'))

print('Estado Psi-:')
display(psi_minus.draw('latex'))

ESTADOS DE BELL
Estado Phi+:


<IPython.core.display.Latex object>

Estado Phi-:


<IPython.core.display.Latex object>

Estado Psi+:


<IPython.core.display.Latex object>

Estado Psi-:


<IPython.core.display.Latex object>

## 7. Espacio de Hilbert $\mathbb{C}^4$
La base computacional de un sistema de dos qubits se obtiene mediante el **producto tensorial** ($\otimes$) de los vectores de $\mathbb{C}^2$. Esto genera una base de 4 dimensiones: $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$.

In [9]:
# Generamos la base completa
base_C4 = {
    "|00>": zero.tensor(zero),
    "|01>": zero.tensor(one),
    "|10>": one.tensor(zero),
    "|11>": one.tensor(one)
}

for etiqueta, vector in base_C4.items():
    print(f"Estado {etiqueta}:")
    display(vector.draw('latex'))

Estado |00>:


<IPython.core.display.Latex object>

Estado |01>:


<IPython.core.display.Latex object>

Estado |10>:


<IPython.core.display.Latex object>

Estado |11>:


<IPython.core.display.Latex object>

## 8. Sistemas Multiqubit en $\mathbb{C}^8$
Finalmente, construimos vectores en un espacio de 8 dimensiones (3 qubits).

In [10]:
# Creamos |000> y |111> usando el producto tensorial
z3 = zero.tensor(zero).tensor(zero)
o3 = one.tensor(one).tensor(one)

# Definimos los vectores
# 1. 1/sqrt(2)|000> + 1/sqrt(2)|111>
v1 = 1/np.sqrt(2) * z3 + 1/np.sqrt(2) * o3

# 2. i/sqrt(2)|000> - 1/sqrt(2)|111>
v2 = 1j/np.sqrt(2) * z3 - 1/np.sqrt(2) * o3

# 3. 1/2|000> + i/2|111>
v3 =1/np.sqrt(2) * z3 + 1j/np.sqrt(2) * o3

# Visualización en LaTeX
print("Vector 1:")
display(v1.draw('latex'))

print("Vector 2:")
display(v2.draw('latex'))

print("Vector 3:")
display(v3.draw('latex'))

Vector 1:


<IPython.core.display.Latex object>

Vector 2:


<IPython.core.display.Latex object>

Vector 3:


<IPython.core.display.Latex object>

## Bibliografía y Referencias
* Qiskit contributors (2023). Qiskit: An Open-source Framework for Quantum Computing.
* Oliphant, T. E. (2006). A guide to NumPy. Trelgol Publishing.
* Nielsen, M. A., & Chuang, I. L. (2010). Quantum Computation and Quantum Information. Cambridge University Press.